# FrameWeave Hybrid v6.0 — HERBench lite_v2 oracle eval (Vanilla + DTR + Hybrid + DTR+Hybrid)

Notebook **chỉ eval**. Không train. Residual injection V5.1 (in-place $\widetilde X$), không join tokens.

Checkpoint FrameWeave Hybrid v6.0 đã train (`lite`). Cây extract **giống** `make_output_tree` + `main_train` lite:

```
outputs_reljoin_v6_videollava/
  checkpoints/best_external_val/   ← eval dùng folder này
    Hybrid.pt                      ← joint Hybrid
    GlobalTimeFix.pt               ← optional ablation
    ...
  checkpoints/last/                ← fallback nếu thiếu best
  logs/  data/  diagnostics/  train_summary.json  config.json
```

Backbone Video-LLaVA-7B đóng băng. Oracle 8-frame giống `DTR_HERBench_Oracle_Full.ipynb`.
Default methods: **`vanilla,dtr,hybrid,dtr_hybrid`**. DTR is re-run on the same oracle clips (not a Drive copy of an old DTR CSV). `dtr_hybrid` loads the same `Hybrid.pt` and turns DTR attention on (layers 18–31) for that generate. Adding this method changes the CSV fingerprint, so a previous `vanilla,dtr,hybrid` run does not resume into the new file.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

Cell staging đọc **đúng path train**: `{extract}/checkpoints/best_external_val/Hybrid.pt`. Không train lại.

## Hardware

Colab **L4 hoặc A100**, disk trống ≥ ~55 GB (Video-LLaVA-7B ~14 GB + HERBench videos ~35 GB). Mount Drive để resume CSV.

## Methods

`vanilla` (hook bypass) + `dtr` (official layers 18–31, residual off) + `hybrid` (`Hybrid.pt`, DTR off) + `dtr_hybrid` (same `Hybrid.pt` and DTR on together). Optional ablations if those `.pt` files exist: `global_timefix`, `hybrid_localoff`, `hybrid_notime`, `hybrid_noglobal`.


## 1. Configuration


In [ ]:
# 1. Eval-only config — không train
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = True          # lần đầu True; smoke OK rồi đặt False trước full (resume CSV)
RUN_FULL = True             # HERBench lite_v2 đầy đủ (~1971 câu)

VARIANT = "Hybrid"
REQUIRED_CKPTS = ["Hybrid.pt"]
METHODS = "vanilla,dtr,hybrid,dtr_hybrid"

EXTRACT_DIR_CANDIDATES = [
    ROOT / "outputs_reljoin_v6_videollava",
    ROOT / "outputs_reljoin_v6_videollava (2)",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_videollava"),
    Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "outputs_reljoin_v6_videollava",
    Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
]

MODEL_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"
DTR_REPO_URL = "https://github.com/ifwub234/DTR.git"
DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"

VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
CKPT_DIR = OUT_DIR / "checkpoints" / "best_external_val"
PRED_DIR = OUT_DIR / "predictions"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
DTR_REPO = ROOT / "DTR"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
EVAL_DIR = ROOT / "herbench_eval_pins"

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_FULL": RUN_FULL,
    "VARIANT": VARIANT,
    "ckpt_name": REQUIRED_CKPTS[0],
    "METHODS": METHODS,
    "REQUIRED_CKPTS": REQUIRED_CKPTS,
    "EXTRACT_DIR_CANDIDATES": [str(p) for p in EXTRACT_DIR_CANDIDATES],
    "CKPT_DIR": str(CKPT_DIR),
    "method": "FrameWeave-Hybrid",
})


## 2. Environment (`dtr_env`)

Python 3.11 + torch 2.0.1+cu118 — cùng pin với notebook DTR.


In [ ]:
# 2. Bootstrap dtr_env (Python 3.11 + torch 2.0.1 như DTR)
from __future__ import annotations

import os
import subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "dtr_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=check)


sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PIP), "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PIP), "install", "--disable-pip-version-check",
    "torch==2.0.1+cu118", "torchvision==0.15.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118",
])
sh([
    str(PIP), "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "hf-transfer", "numpy==1.26.4", "transformers==4.31.0", "tokenizers==0.13.3", "accelerate==0.21.0",
    "datasets==2.21.0", "huggingface_hub==0.24.7", "einops==0.7.0", "sentencepiece==0.2.0", "protobuf<5",
    "av==12.3.0", "pandas==2.2.2", "scipy==1.13.1", "tqdm==4.67.1", "opencv-python-headless==4.10.0.84",
    "Pillow==10.4.0", "pytorchvideo==0.1.5", "fvcore==0.1.5.post20221221", "iopath==0.1.10", "decord==0.6.0",
])
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, torchvision, transformers; "
    "print(torch.__version__); print(torchvision.__version__); print(transformers.__version__); "
    "print(torch.__file__); print(torchvision.__file__); print(transformers.__file__)",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/dtr_env/" in line, f"Package leaked outside venv: {line}"
assert (VENV_BIN / "huggingface-cli").exists()
print("dtr_env ready:", PYTHON)


In [ ]:
!/content/dtr_env/bin/pip install --disable-pip-version-check "peft==0.4.0" "timm==0.6.13" "scikit-learn==1.3.2" "matplotlib==3.8.4"


## 3. Drive, pin DTR, pin HERBench evaluators

Không tải lại evaluator nếu SHA đã khớp (tránh GitHub 429).


In [ ]:
# 3. Drive + clone DTR + pin HERBench evaluator (skip download if SHA matches)
from __future__ import annotations

import hashlib
import os
import subprocess
import time
import urllib.request
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
DTR_REPO = ROOT / "DTR"
DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
EVAL_DIR = ROOT / "herbench_eval_pins"
HERBENCH_BASE_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/model_wrappers/base_vlm.py"
HERBENCH_ACC_URL = "https://raw.githubusercontent.com/DanBenAmi/HERBench/main/evaluation/calculate_accuracy.py"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"
JSDELIVR_BASE = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/model_wrappers/base_vlm.py"
JSDELIVR_ACC = "https://cdn.jsdelivr.net/gh/DanBenAmi/HERBench@main/evaluation/calculate_accuracy.py"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent, EVAL_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DTR_PERSIST = Path("/content/drive/MyDrive/dtr_herbench_oracle_results")
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    DTR_PERSIST = ROOT / "dtr_herbench_oracle_results"
    RELJOIN_PERSIST = OUT_DIR
DTR_PERSIST.mkdir(parents=True, exist_ok=True)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_pinned(urls: list[str], dest: Path, expected: str) -> None:
    if dest.exists() and sha256_file(dest) == expected:
        print("Already pinned:", dest.name)
        return
    last_err = None
    for url in urls:
        for attempt in range(4):
            try:
                print(f"GET {url} (try {attempt + 1})")
                urllib.request.urlretrieve(url, dest)
                got = sha256_file(dest)
                if got == expected:
                    print("OK", dest.name)
                    return
                last_err = RuntimeError(f"SHA mismatch for {dest.name}: {got}")
            except Exception as exc:
                last_err = exc
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Cannot pin {dest.name}: {last_err}")


if not (DTR_REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/ifwub234/DTR.git", str(DTR_REPO)], check=True)
subprocess.run(["git", "-C", str(DTR_REPO), "fetch", "--all"], check=False)
subprocess.run(["git", "-C", str(DTR_REPO), "checkout", DTR_COMMIT], check=True)
got = sha256_file(DTR_REPO / "run_inference_dtr.py")
assert got == DTR_SOURCE_SHA256, f"DTR SHA mismatch: {got}"

fetch_pinned([HERBENCH_BASE_URL, JSDELIVR_BASE], EVAL_DIR / "base_vlm.py", HERBENCH_BASE_SHA256)
fetch_pinned([HERBENCH_ACC_URL, JSDELIVR_ACC], EVAL_DIR / "calculate_accuracy.py", HERBENCH_ACC_SHA256)

print({
    "DTR_REPO": str(DTR_REPO),
    "DTR_PERSIST": str(DTR_PERSIST),
    "RELJOIN_PERSIST": str(RELJOIN_PERSIST),
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "videollava"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave Hybrid sources (eval needs runner + package + DTR oracle helper)
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "run_herbench_oracle_dtr.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "videollava_hooks.py",
    ROOT / "reljoin" / "stats.py",
    ROOT / "reljoin" / "diagnostics.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid eval sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Gắn checkpoint (`outputs_reljoin_v6_videollava/checkpoints`)

Cùng layout `make_output_tree` / `train_variant`:

- **best (eval):** `{extract}/checkpoints/best_external_val/Hybrid.pt`
- **last (fallback):** `{extract}/checkpoints/last/Hybrid.pt`

`--reljoin-ckpt` trỏ thẳng vào `best_external_val`. Default `--methods vanilla,dtr,hybrid,dtr_hybrid`.


In [ ]:
# 5. Bind extract checkpoints — same layout as train.py / make_output_tree
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)

try:
    REQUIRED_CKPTS
    EXTRACT_DIR_CANDIDATES
    METHOD_STEMS
except NameError:
    REQUIRED_CKPTS = ["Hybrid.pt"]
    METHOD_STEMS = {
        "Hybrid": "hybrid",
        "GlobalTimeFix": "global_timefix",
        "HybridLocalOff": "hybrid_localoff",
        "HybridNoTime": "hybrid_notime",
        "HybridNoGlobal": "hybrid_noglobal",
    }
    EXTRACT_DIR_CANDIDATES = [
        ROOT / "outputs_reljoin_v6_videollava",
        ROOT / "outputs_reljoin_v6_videollava (2)",
        Path("/content/drive/MyDrive/outputs_reljoin_v6_videollava"),
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "outputs_reljoin_v6_videollava",
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
    ]

TRAIN_PT_NAMES = ("Hybrid.pt",)


def ckpt_complete(best_dir: Path) -> bool:
    """True if train layout has Hybrid.pt and/or Hybrid.pt."""
    if not best_dir.is_dir():
        return False
    if any((best_dir / n).is_file() for n in REQUIRED_CKPTS):
        return True
    return any((best_dir / n).is_file() for n in TRAIN_PT_NAMES)


def find_extract_root() -> Path:
    req = list(REQUIRED_CKPTS)
    # Find outputs tree: checkpoints/best_external_val/{variant}.pt as in train.py.
    for cand in EXTRACT_DIR_CANDIDATES:
        best = cand / "checkpoints" / "best_external_val"
        if ckpt_complete(best):
            return cand
        last = cand / "checkpoints" / "last"
        if ckpt_complete(last):
            print("WARN: using checkpoints/last (best_external_val incomplete) at", cand)
            return cand
        if any((cand / n).is_file() for n in req):
            return cand
    zip_roots = [
        ROOT,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
    ]
    for base in zip_roots:
        if not base.exists():
            continue
        for zpath in list(base.glob("*reljoin*.zip")) + list(base.glob("*output*.zip")):
            if not zpath.is_file():
                continue
            dest = Path("/tmp/vast_outputs_reljoin_v6_videollava")
            if dest.exists():
                shutil.rmtree(dest)
            print("Extracting", zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            for cand in [dest, *dest.iterdir()]:
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "best_external_val"):
                    return cand
                if cand.is_dir() and ckpt_complete(cand / "checkpoints" / "last"):
                    print("WARN: zip extract using checkpoints/last at", cand)
                    return cand
    scanned = "\n".join(
        f"  {c} exists={c.exists()} best={list((c / 'checkpoints' / 'best_external_val').glob('*.pt')) if (c / 'checkpoints' / 'best_external_val').exists() else 'NO DIR'}"
        for c in EXTRACT_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "Không thấy checkpoint đúng layout train:\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "Upload cả folder `outputs_reljoin_v6_videollava` (kèm checkpoints/) lên /content hoặc Drive.\n"
        f"Đã quét:\n{scanned}"
    )


EXTRACT_DIR = find_extract_root()
BEST_DIR = EXTRACT_DIR / "checkpoints" / "best_external_val"
LAST_DIR = EXTRACT_DIR / "checkpoints" / "last"
if ckpt_complete(BEST_DIR):
    CKPT_DIR = BEST_DIR
elif ckpt_complete(LAST_DIR):
    CKPT_DIR = LAST_DIR
else:
    loose = next((EXTRACT_DIR / n for n in REQUIRED_CKPTS if (EXTRACT_DIR / n).is_file()), None)
    CKPT_DIR = EXTRACT_DIR if loose is not None else BEST_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

off_pt = CKPT_DIR / "Hybrid.pt"
full_pt = CKPT_DIR / "Hybrid.pt"
last_variant = None
summary_path = EXTRACT_DIR / "train_summary.json"
if summary_path.exists():
    try:
        _sm = json.loads(summary_path.read_text(encoding="utf-8"))
        last_variant = _sm.get("last_variant") if isinstance(_sm, dict) else None
        print("train_summary:", _sm)
    except (OSError, json.JSONDecodeError) as exc:
        print("WARN: train_summary.json:", exc)
VARIANT = "Hybrid"
REQUIRED_CKPTS = ["Hybrid.pt"]
METHODS = "vanilla,dtr,hybrid,dtr_hybrid"

resolved = {}
for name in REQUIRED_CKPTS:
    src = CKPT_DIR / name
    if not src.is_file():
        alt = LAST_DIR / name
        if alt.is_file():
            print(f"best missing {name} → copy from last/")
            shutil.copy2(alt, CKPT_DIR / name)
            src = CKPT_DIR / name
        else:
            resolved[name] = None
            continue
    mb = src.stat().st_size / 1e6
    print(f"OK {name}: {src} ({mb:.1f} MB)")
    resolved[name] = src

missing = [n for n, p in resolved.items() if p is None]
assert not missing, f"Thiếu checkpoint: {missing} dưới {CKPT_DIR} và {LAST_DIR}"

if CKPT_DIR.resolve() != LAST_DIR.resolve():
    for variant in METHOD_STEMS:
        name = f"{variant}.pt"
        if not (CKPT_DIR / name).is_file() and (LAST_DIR / name).is_file():
            print(f"missing {name} → copy from last/")
            shutil.copy2(LAST_DIR / name, CKPT_DIR / name)

found_methods = ["vanilla"]
for variant, method in METHOD_STEMS.items():
    path = CKPT_DIR / f"{variant}.pt"
    if path.is_file():
        if path.name not in REQUIRED_CKPTS:
            mb = path.stat().st_size / 1e6
            print(f"OK {path.name}: {path} ({mb:.1f} MB) (not in METHODS)")
        found_methods.append(method)
print("METHODS pinned to", METHODS, "(train variant only)")

for aux in ["train_summary.json", "config.json", "environment.txt"]:
    src = EXTRACT_DIR / aux
    if src.exists() and src.resolve() != (OUT_DIR / aux).resolve():
        shutil.copy2(src, OUT_DIR / aux)
        print("Copied", aux)
log_src = EXTRACT_DIR / "logs" / "train_metrics.csv"
if log_src.exists():
    (OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
    dst = OUT_DIR / "logs" / "train_metrics.csv"
    if log_src.resolve() != dst.resolve():
        shutil.copy2(log_src, dst)

print({
    "EXTRACT_DIR": str(EXTRACT_DIR),
    "CKPT_DIR": str(CKPT_DIR),
    "VARIANT": VARIANT,
    "METHODS": METHODS,
    "pts": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
})


## 6. Tải Video-LLaVA-7B + HERBench lite videos (~35 GB)

PerceptionTest **không** tải — không train.


In [ ]:
# 6. Download Video-LLaVA-7B + HERBench lite (eval only)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'dtr_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
HERBENCH_HF_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_HF_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"

for p in [HERBENCH_ROOT, HERBENCH_VIDEOS, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Downloading Video-LLaVA-7B...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists()

print("Downloading HERBench lite (~35GB)...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{HERBENCH_HF_ID}',
    repo_type='dataset',
    revision='{HERBENCH_REVISION}',
    local_dir=r'{HERBENCH_ROOT}',
    allow_patterns=[
        'data/herbench_lite_v2.parquet',
        'data/herbench_annotations_lite_v2.json',
        'data/*metadata.json',
        'videos/videos.tar.part.00',
        'videos/videos.tar.part.01',
        'videos/videos.tar.part.02',
        'videos/videos.tar.part.03',
        'videos/videos_lite_info.txt',
        'videos/videos.tar.checksums.txt',
    ],
)
print('herbench snapshot OK')
"""], check=True)

LITE_MP4_MIN = 68
parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
print("parts:", [p.name for p in parts])
mp4s = list(HERBENCH_VIDEOS.rglob("*.mp4"))
assert len(parts) >= 4 or len(mp4s) >= 2, (
    f"Thiếu videos.tar.part.00–03 trong {HERBENCH_VIDEOS}. Có: {[p.name for p in parts]}"
)
if len(mp4s) < LITE_MP4_MIN:
    assert len(parts) >= 4, (
        f"Chỉ có {len(mp4s)} mp4 và thiếu tar parts — tải lại videos.tar.part.00–03."
    )
    cat = subprocess.Popen(["cat", *map(str, parts)], stdout=subprocess.PIPE)
    subprocess.run(["tar", "-xf", "-", "-C", str(HERBENCH_VIDEOS)], stdin=cat.stdout, check=True)
    cat.wait()
    print("Extracted archives")
else:
    print(f"Already have {len(mp4s)} mp4")

n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4")))
if n_mp4 >= LITE_MP4_MIN:
    for part in parts:
        part.unlink(missing_ok=True)
    print("Removed archives after full extract")
else:
    print(f"KEEP tar parts — only {n_mp4} mp4 (need ~{LITE_MP4_MIN} for full eval)")

print({
    "videos_mp4": n_mp4,
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert n_mp4 >= 2, "Lite extract xong vẫn thiếu mp4"
if n_mp4 < LITE_MP4_MIN:
    print(f"WARN: {n_mp4} mp4 < {LITE_MP4_MIN}. Smoke có thể chạy; full eval sẽ thiếu video.")
print("Downloads OK")


## 7. Smoke (2 câu) rồi eval full

Smoke **phải** in `Loaded FrameWeave ckpt .../Hybrid.pt` hoặc `.../Hybrid.pt` (đúng `VARIANT`). Nếu `WARNING: no ckpt` thì dừng.

Eval chạy **process tách**. Worker ghi CSV local. `--methods vanilla,dtr,hybrid,dtr_hybrid`: `hybrid` và `dtr_hybrid` cùng dùng `Hybrid.pt`; `dtr_hybrid` bật thêm DTR attention.

Nếu job cũ treo: `!pkill -f run_herbench_oracle_reljoin` rồi xóa `/content/eval_reljoin.job.json`.


In [ ]:
# 7. HERBench oracle eval — smoke then full (NO training; DTR is method dtr on the same clips)
from __future__ import annotations

import json
import os
import shlex
import shutil
import signal
import subprocess
import threading
import time
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
RUNNER = ROOT / "run_herbench_oracle_reljoin.py"
VENV = ROOT / "dtr_env"
DTR_REPO = ROOT / "DTR"
HERBENCH_ROOT = ROOT / "HERBench_official"
HERBENCH_VIDEOS = HERBENCH_ROOT / "videos"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
PRED_DIR = OUT_DIR / "predictions"
EVAL_DIR = ROOT / "herbench_eval_pins"
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (OUT_DIR / "checkpoints" / "best_external_val")
PRED_DIR.mkdir(parents=True, exist_ok=True)
JOB_FILE = ROOT / "eval_reljoin.job.json"
# Worker must persist locally. Drive I/O can stall overnight and freeze eval.
WORKER_PERSIST = PRED_DIR
DRIVE_DIR = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")

def _from_cell1(name: str, default):
    """Read config from the Colab/IPython user namespace (cell 1), not globals()."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    if name in ns:
        return ns[name]
    return default

SMOKE_FIRST = bool(_from_cell1("SMOKE_FIRST", True))
RUN_FULL = bool(_from_cell1("RUN_FULL", True))
VARIANT = "Hybrid"
METHODS = str(_from_cell1("METHODS", "vanilla,dtr,hybrid,dtr_hybrid") or "vanilla,dtr,hybrid,dtr_hybrid")

DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"
HERBENCH_BASE_SHA256 = "6104884c366906d5c223cc9727c3bad778dcdf98c881c4b17b556c20e940af97"
HERBENCH_ACC_SHA256 = "eb67b618f3004b52c8290b2fd30e4012153a66f3f93d576c29760280caa3344d"

needed = list(_from_cell1("REQUIRED_CKPTS", [f"{VARIANT}.pt"]))
if not needed:
    needed = ["Hybrid.pt"]
missing = [n for n in needed if not (CKPT / n).exists()]
assert not missing, (
    f"Chạy cell 5 trước. Thiếu {missing} trong {CKPT}. "
    "Upload folder `outputs_reljoin_v6_videollava/checkpoints/best_external_val/` lên /content."
)

primary_pt = CKPT / needed[0]
n_mp4 = len(list(HERBENCH_VIDEOS.rglob("*.mp4"))) if HERBENCH_VIDEOS.exists() else 0
print({
    "VARIANT": VARIANT,
    "METHODS": METHODS,
    "ckpt": str(primary_pt),
    "ckpt_MB": round(primary_pt.stat().st_size / 1e6, 1) if primary_pt.is_file() else None,
    "videos_mp4": n_mp4,
    "worker_persist": str(WORKER_PERSIST),
    "drive_dir": str(DRIVE_DIR) if DRIVE_DIR.exists() else None,
    "PYTHONPATH": os.environ.get("PYTHONPATH"),
})
assert n_mp4 >= 2, f"Chạy cell 6 trước — chỉ có {n_mp4} mp4 trong {HERBENCH_VIDEOS}."

site = next((VENV / "lib").glob("python3.*/site-packages"))
base_cmd = [
    str(PYTHON), str(RUNNER),
    "--root", "/content", "--venv-site", str(site),
    "--dtr-repo", str(DTR_REPO),
    "--dtr-commit", DTR_COMMIT,
    "--dtr-sha256", DTR_SOURCE_SHA256,
    "--herbench-root", str(HERBENCH_ROOT),
    "--videos-root", str(HERBENCH_VIDEOS),
    "--model-dir", str(MODEL_DIR),
    "--out-dir", str(PRED_DIR),
    "--persist-dir", str(WORKER_PERSIST),
    "--herbench-revision", HERBENCH_REVISION,
    "--model-revision", MODEL_REVISION,
    "--herbench-base-vlm", str(EVAL_DIR / "base_vlm.py"),
    "--herbench-accuracy", str(EVAL_DIR / "calculate_accuracy.py"),
    "--herbench-base-sha256", HERBENCH_BASE_SHA256,
    "--herbench-acc-sha256", HERBENCH_ACC_SHA256,
    "--methods", METHODS,
    "--reljoin-ckpt", str(CKPT),
]
print("eval command methods:", METHODS)


def eval_env() -> dict:
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    return env


def best_effort_copy(src: Path, dst: Path, timeout: float = 8.0) -> bool:
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    try:
        if dst.exists() and src.resolve() == dst.resolve():
            return True
    except OSError:
        return False
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        print(f"WARN: mkdir {dst.parent}: {exc!r}", flush=True)
        return False
    err: list = []

    def _copy() -> None:
        try:
            tmp = dst.with_name(dst.name + ".tmp")
            shutil.copy2(src, tmp)
            tmp.replace(dst)
        except BaseException as exc:
            err.append(exc)

    t = threading.Thread(target=_copy, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        print(f"WARN: copy timed out {timeout}s: {src} -> {dst}", flush=True)
        return False
    if err:
        print(f"WARN: copy failed: {err[0]!r}", flush=True)
        return False
    return True


def pull_drive_csvs() -> None:
    if not DRIVE_DIR.is_dir():
        return
    n = 0
    for src in list(DRIVE_DIR.glob("predictions_reljoin_*.csv")) + list(DRIVE_DIR.glob("summary_reljoin_*.json")):
        if best_effort_copy(src, PRED_DIR / src.name):
            n += 1
    if n:
        print(f"Pulled {n} artifact(s) from Drive -> {PRED_DIR}", flush=True)


def push_drive_csvs() -> None:
    if not DRIVE_DIR.is_dir():
        return
    n = 0
    for src in list(PRED_DIR.glob("predictions_reljoin_*.csv")) + list(PRED_DIR.glob("summary_reljoin_*.json")):
        if best_effort_copy(src, DRIVE_DIR / src.name):
            n += 1
    if n:
        print(f"Pushed {n} artifact(s) -> {DRIVE_DIR}", flush=True)
    else:
        print(f"Drive push skipped or timed out — local CSV remains in {PRED_DIR}", flush=True)


def _pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def _read_job() -> dict | None:
    if not JOB_FILE.exists():
        return None
    try:
        job = json.loads(JOB_FILE.read_text(encoding="utf-8"))
        pid = int(job["pid"])
    except Exception:
        return None
    if not _pid_alive(pid):
        JOB_FILE.unlink(missing_ok=True)
        return None
    return job


def _tail_log(log_path: Path, last_size: int) -> int:
    try:
        data = log_path.read_bytes()
    except OSError:
        return last_size
    if len(data) > last_size:
        print(data[last_size:].decode("utf-8", errors="replace"), end="", flush=True)
        return len(data)
    return last_size


def _kill_pg(pid: int) -> None:
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(pid, sig)
        except OSError:
            try:
                os.kill(pid, sig)
            except OSError:
                return
        if sig == signal.SIGTERM:
            time.sleep(2)


def _log_has_done(log_path: Path, label: str) -> bool:
    mark = "SMOKE OK" if label == "SMOKE" else "FULL OK"
    try:
        return mark in log_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return False


def _wait_job(job: dict) -> bool:
    """True if worker exited. False if this cell was interrupted (worker still runs)."""
    pid = int(job["pid"])
    log_path = Path(job["log"])
    exit_path = Path(job["exit"])
    label = str(job.get("label") or "")
    last_size = 0
    silent_s = 0
    print(f"Waiting pid={pid} label={label} log={log_path}", flush=True)
    try:
        while True:
            prev = last_size
            last_size = _tail_log(log_path, last_size)
            if last_size == prev:
                silent_s += 15
                print(
                    f"[{time.strftime('%H:%M:%S')}] still running pid={pid} "
                    f"log={last_size}B silent={silent_s}s",
                    flush=True,
                )
            else:
                silent_s = 0
            # Smoke/full already printed OK but CUDA teardown hung — kill and continue.
            if _log_has_done(log_path, label) and (not _pid_alive(pid) or silent_s >= 45):
                if _pid_alive(pid):
                    print(
                        f"{label} log already done; killing pid={pid} (CUDA exit hang).",
                        flush=True,
                    )
                    _kill_pg(pid)
                    time.sleep(1)
                if not exit_path.exists():
                    exit_path.write_text("0\n", encoding="utf-8")
                break
            if not _pid_alive(pid):
                time.sleep(1.0)
                last_size = _tail_log(log_path, last_size)
                break
            if silent_s >= 720:
                print(
                    "WARN: log silent >= 12 min (not a done marker). "
                    "Worker may be stuck; local CSV is still valid.",
                    flush=True,
                )
                silent_s = 0
            time.sleep(15)
    except KeyboardInterrupt:
        print(
            f"\nCell interrupted — {label} vẫn chạy pid={pid}. "
            f"Chạy lại cell này để reattach. Log: {log_path}",
            flush=True,
        )
        return False
    last_size = _tail_log(log_path, last_size)
    JOB_FILE.unlink(missing_ok=True)
    code = 1
    if exit_path.exists():
        try:
            code = int(exit_path.read_text(encoding="utf-8").strip() or "1")
        except ValueError:
            code = 1
    elif _log_has_done(log_path, label):
        code = 0
    if code != 0:
        raise RuntimeError(
            f"{label} exited {code}. "
            f"Xem {log_path}. "
            "venv torch / missing videos / state_dict / WARNING: no ckpt. "
            f"videos_mp4={n_mp4} (full cần ~68)."
        )
    return True


def run_eval(cmd: list[str], label: str) -> bool:
    live = _read_job()
    if live is not None:
        print(
            f"Job đang chạy ({live.get('label')} pid={live['pid']}) — reattach, không start mới.",
            flush=True,
        )
        return _wait_job(live)

    log_path = ROOT / f"eval_reljoin_{label.lower()}.log"
    exit_path = ROOT / f"eval_reljoin_{label.lower()}.exit"
    exit_path.unlink(missing_ok=True)
    print("$", " ".join(cmd), flush=True)
    print("detached log:", log_path, flush=True)
    inner = " ".join(shlex.quote(c) for c in cmd)
    script = (
        f"{inner} > {shlex.quote(str(log_path))} 2>&1; "
        f"echo $? > {shlex.quote(str(exit_path))}"
    )
    proc = subprocess.Popen(
        ["bash", "-c", script],
        cwd="/content",
        env=eval_env(),
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    job = {
        "pid": proc.pid,
        "label": label,
        "log": str(log_path),
        "exit": str(exit_path),
    }
    JOB_FILE.write_text(json.dumps(job), encoding="utf-8")
    print(f"started pid={proc.pid} (Stop cell không giết job)", flush=True)
    return _wait_job(job)


def _after_success(label: str) -> None:
    push_drive_csvs()
    if label == "SMOKE":
        print("SMOKE xong — kiểm tra log phải có 'Loaded FrameWeave ckpt'.", flush=True)
    elif label == "FULL":
        print("FULL eval xong. CSV/summary trong", PRED_DIR, flush=True)


pull_drive_csvs()

live = _read_job()
if live is not None:
    ok = _wait_job(live)
    if not ok:
        print("Re-run cell 7 để reattach (đừng start job thứ hai).")
    else:
        _after_success(str(live.get("label") or ""))
        if live.get("label") == "SMOKE" and RUN_FULL:
            if n_mp4 < 68:
                print(f"WARN: {n_mp4} mp4 < 68 — full eval sẽ fail nếu thiếu video; chạy lại cell 6.")
            if run_eval(list(base_cmd), "FULL"):
                _after_success("FULL")
            else:
                print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
else:
    smoke_ok = True
    if SMOKE_FIRST:
        smoke_ok = run_eval(base_cmd + ["--smoke"], "SMOKE")
        if smoke_ok:
            _after_success("SMOKE")
        else:
            print("Smoke vẫn chạy nền. Re-run cell 7 để reattach (đừng start full song song).")
    if smoke_ok and RUN_FULL:
        if n_mp4 < 68:
            print(f"WARN: {n_mp4} mp4 < 68 — full eval sẽ fail nếu thiếu video; chạy lại cell 6.")
        if run_eval(list(base_cmd), "FULL"):
            _after_success("FULL")
        else:
            print("Full vẫn chạy nền. Re-run cell 7 để reattach.")
    elif not RUN_FULL:
        print("RUN_FULL=False — chỉ smoke.")


## 8. Kết quả Vanilla / DTR / Hybrid / DTR+Hybrid

Paired stats in `statistical_tests.json`: `vanilla_vs_dtr`, `vanilla_vs_hybrid`, `vanilla_vs_dtr_hybrid`, `dtr_vs_hybrid`, `dtr_vs_dtr_hybrid`, `hybrid_vs_dtr_hybrid`. DTR and `dtr_hybrid` are produced in this notebook on the same oracle clips.


In [ ]:
# 8. Summarize FrameWeave Hybrid full eval + prior DTR if present
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_videollava"
PRED = OUT / "predictions"
OUT.mkdir(parents=True, exist_ok=True)

dtr_dirs = [
    Path("/content/drive/MyDrive/dtr_herbench_oracle_results"),
    ROOT / "dtr_herbench_oracle_results",
]
dtr_local = next((d for d in dtr_dirs if d.exists()), None)
if dtr_local:
    summaries = sorted(dtr_local.glob("summary_oracle_*.json"), key=lambda p: p.stat().st_mtime)
    if summaries:
        payload = json.loads(summaries[-1].read_text())
        print("Optional extra DTR Drive summary (this notebook already runs method dtr):", summaries[-1])
        if "summary" in payload and "overall" in payload["summary"]:
            display(pd.DataFrame([payload["summary"]["overall"]]))
        if "summary" in payload and "by_task" in payload["summary"]:
            display(pd.DataFrame(payload["summary"]["by_task"]).T)
    else:
        print("Drive DTR folder exists but no summary_oracle_*.json")
else:
    print("No extra Drive DTR folder — use method=dtr rows from this run.")

metrics = OUT / "logs" / "train_metrics.csv"
if not metrics.exists() and "EXTRACT_DIR" in dir():
    metrics = Path(EXTRACT_DIR) / "logs" / "train_metrics.csv"
if metrics.exists():
    print("Lite train_metrics (external PerceptionTest, not HERBench):")
    display(pd.read_csv(metrics))
else:
    print("No train_metrics.csv staged.")

rel_summaries = sorted(PRED.glob("summary_*.json"), key=lambda p: p.stat().st_mtime) if PRED.exists() else []
sci = {
    "herbench_over_vanilla": {"ok": False, "notes": "PENDING paired CI"},
    "paired_ci_promising": {"ok": False, "notes": "PENDING statistical_tests.json"},
}
if rel_summaries:
    rel = json.loads(rel_summaries[-1].read_text())
    print("FrameWeave Hybrid HERBench summary:", rel_summaries[-1].name)
    print(json.dumps(rel.get("summary", {}), indent=2)[:6000])
    inter = rel.get("interventions") or {}
    stats = rel.get("stats") or {}
    if "vanilla_vs_hybrid" in stats:
        boot = stats["vanilla_vs_hybrid"].get("bootstrap") or {}
        sci["herbench_over_vanilla"] = {
            "ok": boot.get("ci_low", -1) > 0,
            "notes": f"delta={boot.get('delta')} CI=({boot.get('ci_low')},{boot.get('ci_high')})",
        }
        sci["paired_ci_promising"] = dict(sci["herbench_over_vanilla"])
    if "vanilla_vs_hybrid_relationoff" in stats:
        boot = stats["vanilla_vs_hybrid_relationoff"].get("bootstrap") or {}
        print("vanilla vs reljoin_relationoff (train RelJoin-Off):", boot)
        sci["vanilla_vs_hybrid_relationoff"] = boot
        if "vanilla_vs_hybrid" not in stats:
            sci["herbench_over_vanilla"] = {
                "ok": boot.get("ci_low", -1) > 0,
                "notes": f"relationoff delta={boot.get('delta')} CI=({boot.get('ci_low')},{boot.get('ci_high')})",
            }
    if inter.get("relation_off") and "reljoin_vs_relation_off" in stats:
        print("reljoin vs relation_off:", stats["reljoin_vs_relation_off"].get("bootstrap") or {})
        sci["reljoin_vs_relation_off"] = stats["reljoin_vs_relation_off"].get("bootstrap") or {}
    for key, payload in stats.items():
        if key.startswith("vanilla_vs_") and key != "vanilla_vs_hybrid":
            print(key, payload.get("bootstrap") or {})
    for key in ("dtr_vs_hybrid", "dtr_vs_dtr_hybrid", "hybrid_vs_dtr_hybrid"):
        if key in stats:
            print(key, stats[key].get("bootstrap") or {})
    stats_path = PRED / "statistical_tests.json"
    if stats_path.exists():
        print("statistical_tests.json:")
        print(stats_path.read_text()[:4000])
else:
    print("Chưa có summary_*.json — chạy cell 7 full trước.")

pairs = sorted(PRED.glob("predictions_reljoin_*.csv"), key=lambda p: p.stat().st_mtime) if PRED.exists() else []
if pairs:
    df = pd.read_csv(pairs[-1])
    ok = df[df["status"] != "runtime_error"] if "status" in df.columns else df
    print("Pred CSV:", pairs[-1].name, "rows", len(df))
    if "method" in ok.columns and "is_correct" in ok.columns:
        display(ok.groupby("method")["is_correct"].mean().rename("accuracy").to_frame())

report = {"scientific": sci, "pred_summaries": [str(p) for p in rel_summaries[-3:]]}
(OUT / "eval_full_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(sci, indent=2))


## 9. Copy artifact sang Drive


In [ ]:
# 9. Persist predictions/summaries to Drive
from __future__ import annotations

import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
PRED = ROOT / "outputs_reljoin_v6_videollava" / "predictions"
PERSIST = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")
if not PERSIST.exists():
    print("Drive persist không có — bỏ qua copy.")
else:
    dest = PERSIST / "eval_full"
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    if PRED.exists():
        for p in PRED.glob("*"):
            if p.is_file():
                shutil.copy2(p, dest / p.name)
                n += 1
    stamp = dest / "copied_at.txt"
    stamp.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
    print(f"Copied {n} files → {dest}")


In [ ]:
# Zip HERBench eval results and download to your laptop
from __future__ import annotations

import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import files

ROOT = Path("/content")
OUT = ROOT / "outputs_reljoin_v6_videollava"
PRED = OUT / "predictions"
DRIVE = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "eval_full"

# Include: predictions CSVs/JSONs + eval_full_report.json (+ Drive copy if present)
# Skip: checkpoints/*.pt and oracle_clips/ (cached frames, can be hundreds of MB)
skip_dirs = {"oracle_clips", "checkpoints"}
skip_suffixes = {".pt", ".bin", ".safetensors"}

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"reljoin_residual_eval_{stamp}.zip"

n = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    extra = [OUT / "eval_full_report.json"]
    if PRED.exists():
        for p in PRED.rglob("*"):
            if not p.is_file():
                continue
            if any(part in skip_dirs for part in p.relative_to(PRED).parts):
                continue
            if p.suffix.lower() in skip_suffixes:
                continue
            zf.write(p, arcname=f"predictions/{p.relative_to(PRED)}")
            n += 1
    for p in extra:
        if p.is_file():
            zf.write(p, arcname=p.name)
            n += 1
    if DRIVE.exists():
        for p in DRIVE.glob("*"):
            if p.is_file() and p.suffix.lower() not in skip_suffixes:
                zf.write(p, arcname=f"drive_eval_full/{p.name}")
                n += 1

print(f"Packed {n} files → {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))
